# Second Document Profile: IDSP Weekly Outbreak Report

This notebook demonstrates automatic profile detection, marker-bounded multi-page table stitching, outbreak-specific row reconstruction, analysis-ready outputs, and profile-specific English questions.

## 1. Project setup and imports

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

working_directory = Path.cwd()
project_root = (
    working_directory
    if (working_directory / "app").exists()
    else working_directory.parent
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.agent.builtin_semantics import BUILTIN_SEMANTIC_CATALOGS
from app.agent.natural_language import NaturalLanguageTableAgent
from app.pipeline.builtin_profiles import BUILTIN_DOCUMENT_PROFILES
from app.pipeline.runner import detect_document_profile, run_profiled_pipeline

sample_documents = project_root / "sample_documents"
outputs = project_root / "outputs"
print("Project root:", project_root)

Project root: C:\Users\vinee\document-table-agent


## 2. Detect every sample document profile

In [2]:
pdf_files = sorted(sample_documents.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError("No PDF files found in sample_documents.")

detected_documents = [
    {
        "PDF": path.name,
        "Profile": detect_document_profile(
            path, BUILTIN_DOCUMENT_PROFILES
        ).name,
    }
    for path in pdf_files
]
display(pd.DataFrame(detected_documents))
outbreak_pdf = next(
    path
    for path in pdf_files
    if detect_document_profile(path, BUILTIN_DOCUMENT_PROFILES).name
    == "idsp_weekly_outbreak_report"
)
print("Using outbreak PDF:", outbreak_pdf.name)

,PDF,Profile
0,75788759701752062509.pdf,idsp_weekly_outbreak_report
1,Weekly 300326 to 050426_544 (1).pdf,grid_india_weekly_report


Using outbreak PDF: 75788759701752062509.pdf


## 3. Run the second profile and export its CSVs

In [3]:
pipeline_result = run_profiled_pipeline(
    outbreak_pdf,
    BUILTIN_DOCUMENT_PROFILES,
    output_dir=outputs,
    overwrite=True,
)
summary = []
for table_name, table_result in pipeline_result.tables.items():
    analysis = table_result.postprocessed_tables["analysis"]
    summary.append(
        {
            "Table": table_name,
            "Pages": f"{table_result.page_numbers[0]}-{table_result.page_numbers[-1]}",
            "Raw shape": table_result.raw_table.shape,
            "Clean shape": table_result.transformed_table.shape,
            "Analysis shape": analysis.shape,
        }
    )
display(pd.DataFrame(summary))

,Table,Pages,Raw shape,Clean shape,Analysis shape
0,current_outbreaks,3-14,"(424, 12)","(34, 10)","(34, 10)"
1,late_outbreaks,15-16,"(41, 14)","(5, 9)","(5, 9)"


## 4. Verify record integrity

In [4]:
current = pipeline_result.tables["current_outbreaks"].postprocessed_tables["analysis"]
late = pipeline_result.tables["late_outbreaks"].postprocessed_tables["analysis"]
all_ids = pd.concat([current["Unique_ID"], late["Unique_ID"]], ignore_index=True)

integrity = {
    "current_records": len(current),
    "late_records": len(late),
    "total_records": len(all_ids),
    "unique_ids": all_ids.nunique(),
    "current_missing_cells": int(current.isna().sum().sum()),
    "late_missing_cells": int(late.isna().sum().sum()),
    "current_cases": int(current["Cases"].sum()),
    "late_cases": int(late["Cases"].sum()),
}
display(pd.Series(integrity, name="Value").to_frame())
assert integrity["total_records"] == 39
assert integrity["unique_ids"] == 39
assert integrity["current_missing_cells"] == 0
assert integrity["late_missing_cells"] == 0

,Value
current_records,34
late_records,5
total_records,39
unique_ids,39
current_missing_cells,0
late_missing_cells,0
current_cases,1026
late_cases,114


## 5. Preview current-week and late-reported records

In [5]:
preview_columns = [
    "Unique_ID",
    "State_UT",
    "District",
    "Disease_Illness",
    "Cases",
    "Deaths",
    "Outbreak_Start_Date",
    "Current_Status",
]
print("Current week")
display(current[preview_columns].head())
print("Reported late")
display(late[preview_columns].head())

Current week


,Unique_ID,State_UT,District,Disease_Illness,Cases,Deaths,Outbreak_Start_Date,Current_Status
0,AR/NAM/2025/19/786,Arunachal Pradesh,Namsai,Chickenpox,12,0,2025-05-08,Under Surveillance
1,AS/BAR/2025/19/787,Assam,Barpeta,Fever of Unknown Origin,19,0,2025-05-08,Under Surveillance
2,AS/DAR/2025/19/788,Assam,Darrang,Food Poisoning,71,0,2025-05-08,Under Control
3,AS/DHE/2025/19/789,Assam,Dhemaji,Food Poisoning,45,0,2025-05-06,Under Surveillance
4,AS/KOK/2025/19/790,Assam,Kokrajhar,Chickenpox,8,0,2025-05-06,Under Surveillance


Reported late


,Unique_ID,State_UT,District,Disease_Illness,Cases,Deaths,Outbreak_Start_Date,Current_Status
0,AP/SRI/2025/19/820,Andhra Pradesh,Sri Sathya Sai,Acute Diarrheal Disease,33,0,2025-02-19,Under Control
1,KL/MAL/2025/19/821,Kerala,Malappuram,Hepatitis A,25,0,2025-04-30,Under Surveillance
2,KL/PAL/2025/19/822,Kerala,Palakkad,Dengue,40,0,2025-04-06,Under Surveillance
3,KL/PAT/2025/19/823,Kerala,Pathanamthitta,Dengue,15,0,2025-04-30,Under Surveillance
4,MG/SOU/2025/19/824,Meghalaya,South Garo Hills,Human Rabies,1,1,2025-04-06,Under Surveillance


## 6. Ask profile-specific English questions

In [6]:
query_tables = {
    table_name: table_result.postprocessed_tables["analysis"]
    for table_name, table_result in pipeline_result.tables.items()
}
language_agent = NaturalLanguageTableAgent(
    query_tables,
    BUILTIN_SEMANTIC_CATALOGS[pipeline_result.profile_name],
)
questions = [
    "Which state had the most outbreak cases?",
    "Which disease had the most deaths?",
    "Show cases for Dengue",
    "Show late reported cases by state",
]
for question in questions:
    result = language_agent.ask(question)
    print("Question:", question)
    display(result.answer)

Question: Which state had the most outbreak cases?


,State_UT,Total_Cases
0,Maharashtra,258


Question: Which disease had the most deaths?


,Disease_Illness,Total_Deaths
0,Food Poisoning,5


Question: Show cases for Dengue


,Unique_ID,State_UT,District,Disease_Illness,Cases,Deaths,Outbreak_Start_Date,Reporting_Date,Current_Status,Comments_Action_Taken
0,KL/ERN/2025/19/803,Kerala,Ernakulam,Dengue,21,0,2025-05-06,2025-05-06,Under Surveillance,Cases were reported from Sub-District Kothaman...
1,KL/PAL/2025/19/806,Kerala,Palakkad,Dengue,49,2,2025-05-08,2025-05-09,Under Surveillance,"Cases were reported from Village Chalissery, S..."


Question: Show late reported cases by state


,State_UT,Total_Cases
0,Kerala,80
1,Andhra Pradesh,33
2,Meghalaya,1


## 7. Findings

- The two PDFs are detected as different document families without filename rules.
- Marker-bounded extraction stitches current-week pages 3-14 and late-report pages 15-16.
- Profile-specific preprocessing reconstructs one logical row per outbreak while raw extraction remains unchanged.
- The report contains 34 current-week and 5 late-reported outbreaks, for 39 unique records.
- Dates and numeric measures are typed in analysis outputs, and the second semantic catalog supports outbreak questions.
- The original electricity profile continues to use the same parser, validator, runner, query engine, and CLI.